# Dice por imagen de validación de las 50 corridas (base del ítem 33)

Lee el mejor checkpoint de cada corrida, verifica su SHA-256 contra `result.json`, comprueba que la media por grupos
reproduzca `best_val_group_Dice` y escribe un JSON por corrida. No necesita GPU. Al final descarga
`validacion_por_imagen_e2.zip`, que va a `resultados/validacion_por_imagen_e2/` del repositorio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
RUNS = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')
corridas = sorted(RUNS.glob('*_seed*/result.json'))
print(len(corridas), 'corridas con result.json')
assert len(corridas) == 50, 'Se esperaban 50 corridas'

In [ ]:
import hashlib
SCRIPT = '"""Extrae el Dice por imagen de validación guardado en el mejor checkpoint de cada corrida.\n\nscripts/experimento_e2_colab.py guarda en best_epoch_*.pt la clave \'validation\'\n(lista de {id, group_id, Dice, IoU, mIoU, accuracy}) de la época elegida. Este\nscript la vuelca a un JSON por corrida, verificando el SHA-256 del checkpoint\ncontra result.json y que la media por grupos reproduzca best_val_group_Dice.\nNecesita torch (Colab o el entorno local del proyecto); no entrena ni evalúa.\n\nUso:\n  python extraer_validacion_por_imagen.py --corridas /ruta/experimento --salida resultados/validacion_por_imagen_e2\n"""\nimport argparse\nimport hashlib\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for bloque in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(bloque)\n    return h.hexdigest()\n\n\ndef media_por_grupo(detalle):\n    grupos = defaultdict(list)\n    for fila in detalle:\n        grupos[fila[\'group_id\']].append(fila[\'Dice\'])\n    return float(np.mean([np.mean(v) for v in grupos.values()]))\n\n\ndef extraer(carpeta_corrida):\n    import torch\n    resultado = json.loads((carpeta_corrida / \'result.json\').read_text(encoding=\'utf-8\'))\n    if not resultado.get(\'completed\'):\n        raise RuntimeError(f\'Corrida incompleta: {carpeta_corrida}\')\n    mejor = carpeta_corrida / resultado[\'best_file\']\n    real = sha256(mejor)\n    if real != resultado[\'best_sha256\']:\n        raise RuntimeError(f\'SHA-256 distinto en {mejor}\')\n    guardado = torch.load(mejor, map_location=\'cpu\', weights_only=False)\n    detalle = guardado[\'validation\']\n    media = media_por_grupo(detalle)\n    if abs(media - resultado[\'best_val_group_Dice\']) > 1e-9:\n        raise RuntimeError(f\'La media por grupos no reproduce result.json en {carpeta_corrida}\')\n    return {\'model\': resultado[\'model\'], \'seed\': resultado[\'seed\'], \'split\': \'val\',\n            \'best_file\': resultado[\'best_file\'], \'best_sha256\': real,\n            \'epoch\': int(guardado[\'epoch\']) + 1, \'val_group_Dice\': media,\n            \'detail\': [{k: fila[k] for k in (\'id\', \'group_id\', \'Dice\', \'IoU\')} for fila in detalle]}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--corridas\', required=True, help=\'carpeta con <modelo>_seed<semilla>/result.json\')\n    parser.add_argument(\'--salida\', required=True)\n    args = parser.parse_args()\n    salida = Path(args.salida)\n    salida.mkdir(parents=True, exist_ok=True)\n    hechas = 0\n    for resultado in sorted(Path(args.corridas).glob(\'*_seed*/result.json\')):\n        datos = extraer(resultado.parent)\n        destino = salida / f"{datos[\'model\']}_seed{datos[\'seed\']}_val_por_imagen.json"\n        destino.write_text(json.dumps(datos, indent=1, ensure_ascii=False), encoding=\'utf-8\')\n        hechas += 1\n        print(f"{destino.name}: {len(datos[\'detail\'])} imágenes, Dice grupos {datos[\'val_group_Dice\']:.5f}")\n    if not hechas:\n        raise SystemExit(\'No se encontró ningún result.json\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
assert hashlib.sha256(SCRIPT.encode('utf-8')).hexdigest() == '80b658e04f8f4036a4a43e7bfb1e76cde70474538ecd6f19d0df8cd6cdb264d8'
Path('/content/extraer_validacion_por_imagen.py').write_text(SCRIPT, encoding='utf-8')

In [ ]:
!python /content/extraer_validacion_por_imagen.py --corridas "$RUNS" --salida /content/validacion_por_imagen_e2

In [ ]:
import shutil
from google.colab import files
assert len(list(Path('/content/validacion_por_imagen_e2').glob('*.json'))) == 50
shutil.make_archive('/content/validacion_por_imagen_e2', 'zip', '/content/validacion_por_imagen_e2')
files.download('/content/validacion_por_imagen_e2.zip')